# Two-time correlation processing

This notebook reads the two-time CSV in this folder, builds the complex matrix $C(t',t)$ for a selected center/site, and saves plots and reduced data in `processed_twotime/`.

In [1]:
# Imports and file locations.
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def load_csv_data(csv_file):
    """
    Load the CSV file and return a DataFrame with the specified columns and dtypes.
    """
    # The explicit dtypes keep memory use reasonable for this large CSV.
    columns = ['t_prime', 't', 'center_site', 'site_index', 'real_part', 'imaginary_part']
    dtypes = {
        't_prime': 'float64', 't': 'float64',
        'center_site': 'int16', 'site_index': 'int16',
        'real_part': 'float64', 'imaginary_part': 'float64',
    }
    data = pd.read_csv(csv_file, usecols=columns, dtype=dtypes)
    if data.empty:
        raise ValueError('The CSV contains no data rows.')
    
    return data

def pivot_to_matrix(data, center_site, site_index):
    # Select the rows for the chosen center_site and site_index, and pivot to a matrix.
    selected = data.loc[
        (data.center_site == CENTER_SITE) & (data.site_index == REFERENCE_SITE)
    ].copy()
    if selected.empty:
        raise ValueError(f'No G data for center_site=site_index={CENTER_SITE}')

    # Do not average: each (t_prime, t) pair must occur exactly once.
    duplicates = selected.duplicated(['t_prime', 't'], keep=False)
    if duplicates.any():
        examples = selected.loc[duplicates, ['t_prime', 't']].head(10).to_dict('records')
        raise ValueError(f'Duplicate (t_prime, t) rows found. Examples: {examples}')

    # Pandas 3.13 cannot unstack a complex dtype ("No matching signature found").
    # Pivot float components independently, then construct the complex matrix.
    matrix_real = selected.pivot(index='t_prime', columns='t', values='real_part').sort_index().sort_index(axis=1)
    matrix_imag = selected.pivot(index='t_prime', columns='t', values='imaginary_part').sort_index().sort_index(axis=1)
    if not matrix_real.index.equals(matrix_imag.index) or not matrix_real.columns.equals(matrix_imag.columns):
        raise RuntimeError('Real and imaginary two-time grids do not match.')
    t_prime = matrix_real.index.to_numpy()
    t = matrix_real.columns.to_numpy(dtype=float)
    C = matrix_real.to_numpy() + 1j * matrix_imag.to_numpy()
    missing = np.isnan(C.real).sum()
    print(f'Matrix shape: {C.shape} (rows=t_prime, columns=t); missing entries: {missing}')
    if missing:
        print('Warning: missing pairs are retained as NaN and ignored in averages.')
    
    return t, t_prime, C

# Ggreat_pump_0_csv = load_csv_data(Path(os.getcwd()) / "G_files_dir" / "Ggreat_twotime_td_N=8_Nup=4_Ndown=4_20261017_215433_freq_0_P2_c_P3_centersite=3.csv")

# Gless_pump_0_csv = load_csv_data(Path(os.getcwd()) / "G_files_dir" / "Gless_twotime_td_N=8_Nup=4_Ndown=4_20261017_215433_freq_0_P2_c_P3_centersite=3.csv")



Ggreat_pump_0_csv = load_csv_data(Path(os.getcwd()) / "new_A" / "G>_twotime_td_N=8_Nup=4_Ndown=4_20261019_200741_freq_0_P2_c_P3_centersite=3.csv")

Gless_pump_0_csv = load_csv_data(Path(os.getcwd()) / "new_A" / "G<_twotime_td_N=8_Nup=4_Ndown=4_20261019_200741_freq_0_P2_c_P3_centersite=3.csv")

# Ggreat_pump_0_csv = load_csv_data(Path(os.getcwd()) / "G_files_dir" / "Ggreat_twotime_td_N=8_Nup=4_Ndown=4_20261017_220624_freq_6_P2_c_P3_centersite=3.csv")

# Gless_pump_0_csv = load_csv_data(Path(os.getcwd()) / "G_files_dir" / "Gless_twotime_td_N=8_Nup=4_Ndown=4_20261017_220624_freq_6_P2_c_P3_centersite=3.csv")


In [2]:
# Build G(center_site, center_site; t_prime, t).  Change CENTER_SITE to inspect another center.
CENTER_SITE = 3
REFERENCE_SITE = CENTER_SITE  # Enforces G(center_site, center_site).

t_great, t_prime_great, Ggreat_pump_0_matrix = pivot_to_matrix(Ggreat_pump_0_csv, CENTER_SITE, REFERENCE_SITE)

t_less, t_prime_less, Gless_pump_0_matrix = pivot_to_matrix(Gless_pump_0_csv, CENTER_SITE, REFERENCE_SITE)

def plot_two_time_correlation(t, t_prime, C, center_site, site_index, filename):
    # Two-time maps.  The horizontal axis is t and the vertical axis is t_prime.
    extent = [t.min(), t.max(), t_prime.min(), t_prime.max()]
    views = [(C.real, 'Re C'), (C.imag, 'Im C'), (np.abs(C), '|C|'), (np.angle(C), 'arg(C)')]
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
    for ax, (values, title) in zip(axes.flat, views):
        image = ax.imshow(values, origin='lower', aspect='auto', extent=extent, interpolation='nearest', cmap='viridis')
        ax.set(title=title, xlabel='t', ylabel="t'")
        fig.colorbar(image, ax=ax)
    fig.suptitle(f'Two-time correlation: center={center_site}, site={site_index}')
    plot_file = filename + f'_center={center_site}, site={site_index}.png'
    fig.savefig(plot_file, dpi=200, bbox_inches='tight')
    print(f'Saved {plot_file}')
    plt.show()

# plot_two_time_correlation(t, t_prime, Ggreat_pump_0_matrix, CENTER_SITE, REFERENCE_SITE, filename='Ggreat_Pump_0')
# plot_two_time_correlation(t, t_prime, Gless_pump_0_matrix, CENTER_SITE, REFERENCE_SITE, filename='Gless_Pump_0')

Matrix shape: (1000, 1001) (rows=t_prime, columns=t); missing entries: 499968
Matrix shape: (1001, 1001) (rows=t_prime, columns=t); missing entries: 501116


## Gaussian-windowed Fourier transform (Eq. 27)

Equation (27) of `2109.13444v1.pdf` defines $I(\omega,t_0)=\int dt\,dt' G^<(t,t')s(t-t_0)s(t'-t_0)e^{-i\omega(t-t')}$. The input stores only one triangle, so this cell completes it using Hermiticity of the raw correlator.

In [3]:
def my_FT_singletime(G, time_axis, t_probe, sigma, frequency_axis):
    """Transform G(t_probe, t') using the Perl script's Gaussian and Fourier convention."""
    time_axis, _ = _time_step(time_axis)
    G = np.asarray(G, dtype=np.complex128)
    if G.shape != (len(time_axis), len(time_axis)):
        raise ValueError('G must be square with one row and column per time point.')
    if sigma <= 0:
        raise ValueError('sigma must be positive.')
    index_t_probe = _probe_index(time_axis, t_probe)
    frequencies = np.asarray(frequency_axis, dtype=float)
    valid = np.isfinite(G[index_t_probe, :].real) & np.isfinite(G[index_t_probe, :].imag)
    if not np.all(valid):
        print(f'Warning: omitting {np.count_nonzero(~valid):,} missing single-time G values.')
    signal = np.where(valid, G[index_t_probe, :], 0.0)
    signal *= np.exp(-(time_axis ** 2) / (2 * sigma ** 2))
    # Perl computes sum(value * exp(+i * omega * time)); it applies no dt
    # factor or Gaussian normalization.
    transformed = np.array([
        np.sum(signal * np.exp(1j * frequency * time_axis))
        for frequency in frequencies
    ])
    return -np.imag(transformed) / np.pi

ValueError: Ggreat and Gless time grids differ.

In [ ]:
# Compute and plot the single-time spectrum.
t = t_great
G_retarded = -1j * (Ggreat_pump_0_matrix[:len(t), :len(t)] - Gless_pump_0_matrix[:len(t), :len(t)])
frequency_axis = np.arange(-20, 20, 0.1)
sigma = 5
t_probe = 12
I_omega_td_stc = my_FT_singletime(G_retarded, t, t_probe, sigma, frequency_axis)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(frequency_axis, I_omega_td_stc, label=f'Single-time spectrum, t_probe={t_probe}')
ax.set(xlabel='Frequency (ω)', ylabel='Intensity', title='Single-time Fourier transform')
ax.axhline(0, color='black', linewidth=0.7)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()